# 🛡️ Notebook 4: Timeouts & Graceful Degradation

Two resilience patterns that pair with everything else in this lab:

1. **Timeouts** — the single most important resilience pattern. Without them, a slow downstream becomes an infinite wait, and all the retry/breaker/bulkhead tricks in the world can't save you.
2. **Graceful degradation** — when something *does* fail, serve a *degraded* answer instead of a broken page. 'Slightly worse product' beats '500 Internal Server Error'.

> Netflix motto: *"Better to serve you a stale recommendation than no page at all."*

## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

### 🟥 BAD: no timeout

A downstream that never responds holds our thread forever. One such call is survivable; a stream of them exhausts the thread pool and the whole service goes down. **This is how a partial outage becomes a total one.**

Let's actually run it: a 4-thread server, 4 stuck calls, and then one perfectly healthy request that should take 0 ms.


In [ ]:
import time, threading
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout

# A downstream that has stopped answering. `release` lets us free the threads
# at the end of the cell so the notebook doesn't actually hang for 30s.
release = threading.Event()

def hanging_call():
    release.wait(30)          # never returns on its own
    return 'ok'

def healthy_call():
    return 'healthy page'     # instant — nothing wrong with THIS request

server = ThreadPoolExecutor(max_workers=4)   # our whole request-handling capacity
for _ in range(4):
    server.submit(hanging_call)              # 4 stuck calls -> every thread is gone
time.sleep(0.1)                              # let them grab the threads

t0 = time.perf_counter()
fut = server.submit(healthy_call)            # a healthy user shows up
try:
    print('healthy request returned:', fut.result(timeout=1.0))
except FutureTimeout:
    print(f'healthy request STILL queued after {time.perf_counter() - t0:.1f}s '
          '— the server has no free threads left')
print('-> 4 stuck calls took down 100% of the service. No timeout = no recovery.')

release.set(); server.shutdown(wait=True)    # clean up

### 🟩 GOOD: a hard timeout

Real HTTP clients (`requests`, `httpx`, `aiohttp`) have a `timeout=` argument — *always set it*. Below we show the general idea with a worker thread and `Event.wait()`:

In [ ]:
class TimeoutError_(Exception): pass

def call_with_timeout(fn, timeout):
    """Run fn() in a background thread; raise if it doesn't finish in `timeout` seconds."""
    result, error, done = [], [], threading.Event()

    def worker():
        try: result.append(fn())
        except Exception as e: error.append(e)
        finally: done.set()

    threading.Thread(target=worker, daemon=True).start()
    if not done.wait(timeout):
        raise TimeoutError_(f'call exceeded {timeout}s')
    if error: raise error[0]
    return result[0]

t0 = time.perf_counter()
try:
    call_with_timeout(hanging_call, timeout=0.3)
except TimeoutError_ as e:
    print(f'timed out after {time.perf_counter()-t0:.2f}s:', e)


### ⚠️ A missing timeout defeats every *other* mechanism

This is the point people miss. Retries, circuit breakers and bulkheads all react to a call **finishing badly**. A call that never finishes never produces a failure to react to — so the breaker never trips, the retry never fires, and the bulkhead just fills up and stays full.

Below: the same breaker + retry wrapper around a dependency that answers *far too slowly*, with and without a per-attempt timeout.


In [ ]:
class MiniBreaker:
    """Consecutive-failure breaker (notebook 2), trimmed to the essentials."""
    def __init__(self, threshold=2):
        self.threshold, self.fails, self.state = threshold, 0, 'CLOSED'

    def call(self, fn):
        if self.state == 'OPEN':
            raise RuntimeError('circuit OPEN — fast-fail')
        try:
            r = fn()
        except Exception:
            self.fails += 1
            if self.fails >= self.threshold:
                self.state = 'OPEN'
            raise
        self.fails = 0
        return r

def stalling_dependency():
    """Not dead — just answers 1s late, forever. The worst kind of sick."""
    time.sleep(1.0)
    return 'ok (but 1s too late)'

def drive(call, label, n=4):
    cb = MiniBreaker(threshold=2)
    t0 = time.perf_counter()
    for _ in range(n):
        try:
            cb.call(lambda: call(stalling_dependency))
        except Exception:
            pass
    print(f'{label:<22} elapsed {time.perf_counter() - t0:5.2f}s   breaker={cb.state}')

drive(lambda f: f(), 'NO timeout:')                              # breaker never sees a failure
drive(lambda f: call_with_timeout(f, 0.2), 'WITH 0.2s timeout:')  # breaker gets to do its job

print('\nWithout a timeout the breaker stays CLOSED forever — it has nothing to count.')
print('The timeout is what *converts* "slow" into "failed" so the other patterns can act.')

### 🌍 Real-world rules

- **Always set timeouts** on every network call. `requests.get(url)` with no `timeout=` is a production incident waiting to happen.
- **Pick sensible values** — usually a few multiples of the 99th-percentile latency, *not* a huge 'safe' default like 60 s.
- **Timeouts should shrink as you go deeper** — if the user-facing API has a 2 s budget, the DB call within it should have maybe 500 ms. Otherwise the outer timeout fires while the inner call is still waiting.
- **Caller timeout < server timeout** — if the caller gives up first, the server is still working on a dead request. Coordinate them.

In async Python, use [`asyncio.timeout(...)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.timeout):
```python
async with asyncio.timeout(0.3):
    await call_downstream()
```

## 💔 Part 2 — Graceful degradation

When an optional dependency fails, don't bubble the error to the user. Return something *good enough*:

| Feature | If dependency fails, serve… |
|---|---|
| Personalized recommendations | a generic 'popular products' list |
| User avatar | a default silhouette image |
| Friend activity feed | an empty list, not a crashed page |
| Currency conversion | last-known exchange rate from cache |
| Search with typo-correction | raw search results without correction |

### 🟥 BAD: one failure breaks the page

In [ ]:
def recommendations_service():
    raise IOError('recommendations ML service down')

def product_page_bad(user):
    recs = recommendations_service()   # <- raises
    return {'user': user, 'recommendations': recs}

try:
    page = product_page_bad('alice')
except IOError as e:
    print('page rendering FAILED:', e, '— user sees a 500')


### 🟩 GOOD: fallback to a default

Catch the failure at the edge of the feature and substitute a sensible default. The user still gets a product page — just without personalization.

In [ ]:
POPULAR_FALLBACK = ['socks', 't-shirt', 'mug']   # safe, non-personalized default

def get_recommendations(user):
    try:
        return recommendations_service()
    except Exception as e:
        print(f'  ⚠ recommendations failed ({e}); serving popular-items fallback')
        return POPULAR_FALLBACK

def product_page_good(user):
    return {'user': user, 'recommendations': get_recommendations(user)}

print(product_page_good('alice'))


### 🧊 Even better: stale cache as fallback

Serve the *last good answer* instead of a generic default — users barely notice.

In [ ]:
_cache = {}   # user -> last good recommendations

def get_recs_with_stale_cache(user):
    try:
        fresh = recommendations_service()
        _cache[user] = fresh   # refresh the cache on success
        return fresh
    except Exception:
        if user in _cache:
            print('  ℹ️ serving stale cache (recommendations service is down)')
            return _cache[user]
        print('  ⚠ no cache — falling back to popular items')
        return POPULAR_FALLBACK

# Pretend we had a good response yesterday:
_cache['alice'] = ['running shoes', 'water bottle', 'headband']
print(get_recs_with_stale_cache('alice'))
print(get_recs_with_stale_cache('bob'))


## 🚦 Part 3 — Feature flags (kill switches)

Sometimes graceful degradation has to happen *before* the call even goes out. A **feature flag** (aka *kill switch*) lets on-call operators turn off an expensive feature in seconds without a deploy:

- 'Disable recommendations panel' — service-under-load relief valve.
- 'Disable image thumbnails' — skip the thumbnailer during the outage.
- 'Read-only mode' — turn off writes when the primary DB is in trouble.

Common libraries: [Unleash](https://www.getunleash.io/), [LaunchDarkly](https://launchdarkly.com/), or a simple config table in your database.

In [ ]:
FEATURE_FLAGS = {
    'show_recommendations': True,   # on-call flips this to False during an outage
    'show_avatars': True,
}

def product_page_with_flags(user):
    page = {'user': user}
    if FEATURE_FLAGS['show_recommendations']:
        page['recommendations'] = get_recs_with_stale_cache(user)
    return page

print('feature ON :', product_page_with_flags('alice'))

# On-call disables the feature
FEATURE_FLAGS['show_recommendations'] = False
print('feature OFF:', product_page_with_flags('alice'))


## 🚰 Part 4 — Load shedding

Degradation drops *features*. **Load shedding** drops *requests* — deliberately, early, and cheaply — when you are past the point where you can serve them all.

The counter-intuitive part: a server with no shedding doesn't fail gracefully at overload, it fails *expensively*. It queues everything, every client eventually times out, and the server spends 100% of its CPU producing responses nobody is waiting for any more. That's **goodput collapse**: throughput stays high, useful throughput goes to zero.

Below: 40 clients with a 250 ms deadline hit a server with 4 worker slots and 50 ms of work per request.

In [ ]:
import threading, time

WORKERS, SERVICE, DEADLINE, CLIENTS = 4, 0.05, 0.25, 40

def run_server(max_queue=None, extra_clients=0):
    """max_queue=None -> no shedding (queue everything). Otherwise reject past the cap."""
    sem = threading.Semaphore(WORKERS)
    lock = threading.Lock()
    queued = 0
    s = {'served_in_time': 0, 'client_gave_up': 0, 'shed_503': 0, 'wasted_work_s': 0.0}

    def handle():
        nonlocal queued
        with lock:
            if max_queue is not None and queued >= max_queue:
                s['shed_503'] += 1          # instant 503 + Retry-After: costs ~nothing
                return
            queued += 1
        t0 = time.perf_counter()
        sem.acquire()
        try:
            time.sleep(SERVICE)             # the server ALWAYS does the full work
        finally:
            sem.release()
            with lock: queued -= 1
        latency = time.perf_counter() - t0
        with lock:
            if latency <= DEADLINE:
                s['served_in_time'] += 1
            else:
                s['client_gave_up'] += 1    # client timed out; response goes in the bin
                s['wasted_work_s'] += SERVICE

    ts = [threading.Thread(target=handle) for _ in range(CLIENTS + extra_clients)]
    for t in ts: t.start()
    for t in ts: t.join()
    s['wasted_work_s'] = round(s['wasted_work_s'], 2)
    return s

no_shed = run_server(max_queue=None)
shed    = run_server(max_queue=WORKERS * 3)   # allow a small queue, reject beyond it
print('no shedding  :', no_shed)
print('with shedding:', shed)

Read it honestly: shedding served **fewer** requests in this single round (12 vs 16). What it bought is that nobody waited past their deadline and no CPU was spent on abandoned work.

The decisive difference shows up on the **next** round. The clients that timed out retry immediately; the clients that got a `503` + `Retry-After` back off. So the unshed server takes its own timeouts as extra load:

In [ ]:
# Round 2: everyone who timed out retries right away.
# Shed clients honour Retry-After, so they come back later, not now.
no_shed_r2 = run_server(max_queue=None,          extra_clients=no_shed['client_gave_up'])
shed_r2    = run_server(max_queue=WORKERS * 3,   extra_clients=0)

print(f"round 2, no shedding  : offered {CLIENTS + no_shed['client_gave_up']:>2}  "
      f"served_in_time {no_shed_r2['served_in_time']:>2}  wasted {no_shed_r2['wasted_work_s']}s")
print(f"round 2, with shedding: offered {CLIENTS:>2}  "
      f"served_in_time {shed_r2['served_in_time']:>2}  wasted {shed_r2['wasted_work_s']}s")
print('\nUseful throughput did not move, but wasted work doubled: the unshed server')
print('is now spending most of its capacity on responses nobody is waiting for,')
print('and every round of timeouts feeds the next one. That is the goodput spiral.')

### 🌍 Shedding in practice

- **Shed the cheapest way possible.** A `503` + `Retry-After` written before any DB work costs microseconds; a `503` written after the query costs the query.
- **Shed by priority, not at random.** Checkout > search > recommendations; paying tenant > free tier; a user's first request > their 40th.
- **Pick the signal deliberately** — queue depth (used above), in-flight count, p99 latency, or CPU. Queue depth is the most direct proxy for "we are already late".
- **Never shed health checks or admin endpoints** — you'll get yourself removed from the load balancer *and* lose the ability to fix it.
- **Trade-off:** shedding is a deliberate decision to fail some users *now* so that the rest succeed. If your traffic is not actually over capacity, a shedder that is tuned too tight just invents outages. Ship it with a metric on the shed rate and alert on it.

**When NOT to shed:** background/batch work with no human waiting (queue it instead), and anything where a partial result is worse than a slow one (a half-written transaction).

## ❤️‍🩹 Part 5 — Health checks

A load balancer only knows to stop sending traffic to a broken instance if the instance tells it. That's the health check — and it's the pattern most often implemented backwards.

| Check | Question it answers | Who consumes it | If it fails |
|---|---|---|---|
| **Liveness** | *Is this process wedged?* | orchestrator (Kubernetes) | restart the container |
| **Readiness** | *Should I get traffic right now?* | load balancer | remove from rotation |
| **Startup** | *Has warm-up finished?* | orchestrator | keep waiting, don't kill it |

### 🟥 BAD: a deep readiness check that pings every dependency

It feels thorough. It is a **correlated, fleet-wide outage generator**: when a non-critical dependency blips, *every* instance reports unhealthy at the same moment, the load balancer has nothing left to route to, and a degraded feature becomes a total outage.

In [ ]:
# Three identical instances behind one load balancer.
deps_up = {'database': True, 'recommendations': True}

def deep_readiness():
    """BAD: 'healthy' means EVERY dependency answered."""
    return all(deps_up.values())

def critical_only_readiness():
    """GOOD: 'healthy' means I can still serve a useful response."""
    return deps_up['database']          # recs failure is handled by degradation

def load_balancer(check, instances=3, requests=100):
    healthy = [check() for _ in range(instances)]   # each instance answers for itself
    if not any(healthy):
        return {'healthy_instances': 0, 'served': 0, 'dropped': requests}
    return {'healthy_instances': sum(healthy), 'served': requests, 'dropped': 0}

deps_up['recommendations'] = False      # a NON-critical dependency blips
print('deep check      :', load_balancer(deep_readiness))
print('critical-only   :', load_balancer(critical_only_readiness))
print('\nThe recs outage only had to cost us personalization. The deep health check')
print('turned it into a 100% outage — on every instance simultaneously.')

### 🌍 Health-check rules

- **Readiness checks only *critical* dependencies.** If the request can still be served (even degraded), report healthy. Non-critical dependencies belong to the fallback path, not the health check.
- **Liveness must be shallow.** A liveness probe that touches the database will restart-loop your whole fleet during a DB blip — and restarting never fixes a DB.
- **Give the check its own (short) timeout.** A health check that hangs is a health check that fails, and now your slow dependency is also removing you from rotation.
- **Add hysteresis.** Require N consecutive failures to go unhealthy and M consecutive successes to come back, or you'll flap in and out of rotation.
- **Cache the result** (1–2 s). A per-second probe from every LB node that runs a real query is itself a load source.
- **Fail open at the fleet level.** Many load balancers implement *panic mode*: if more than ~50% of instances report unhealthy, ignore health entirely and spread traffic over all of them — because "everything is unhealthy" is far more likely to mean the check is wrong than that every server died.

## 🧠 Putting it all together

The patterns in this lab stack — each layer handles a *different* failure mode:

```
  user request
      |
      v
  load shedding       <- past capacity? reject NOW, cheaply, with Retry-After
      |
      v
  feature flag        <- (cheap) turn off non-essential features under load
      |
      v
  bulkhead            <- limit concurrent calls per dependency
      |
      v
  circuit breaker     <- short-circuit if dependency is dead
      |
      v
  retry w/ jitter     <- survive transient blips
      |
      v
  timeout             <- never wait forever  (without this, none of the above fire)
      |
      v
  downstream service
```

…and on failure, **graceful degradation** / fallback gives the user a still-useful page. Meanwhile the **readiness check** tells the load balancer whether this instance should be in rotation at all.

| Pattern | Problem it solves | When NOT to use it |
|---|---|---|
| Timeout | a slow call that never returns | never — always set one (but tune it; too tight invents failures) |
| Retry + jitter | transient network blips | non-idempotent writes without an idempotency key; `4xx`; already-overloaded deps |
| Circuit breaker | a sustained outage | low-traffic paths (too few samples to judge); dependencies with no fallback and no alternative |
| Bulkhead | blast radius — one slow dep taking down unrelated ones | a single dependency, or when partitioning leaves each pool too small to be useful |
| Load shedding | more traffic than capacity | background work with no waiting human; when you're under capacity (it only invents outages) |
| Graceful degradation / fallback | still serve *something* when things fail | correctness-critical reads (a balance, a permission check) — a wrong answer is worse than an error |
| Feature flag / kill switch | operator-controlled load shedding | as a substitute for automatic protection — humans are slower than an outage |
| Health check | routing traffic away from a broken instance | as a dependency monitor — that's what alerting is for |
